# mPES — Optimización Bayesiana en Google Colab Pro+

Ejecuta las optimizaciones Bayesianas de **pes_dqn** (DQN) o **pes_ac** (A2C)
en una instancia GPU de Google Colab Pro+.

### Ventajas sobre la máquina local
- **Más RAM** (hasta 83 GB vs 7.7 GB) — sin riesgo de OOM killer
- **Runtime continuo** de hasta 24h
- **CPU más potente** (Xeon vs i3-6006U)

> **Nota sobre GPU:** Los modelos DQN/A2C son pequeños (32-64 neuronas, 1-2 capas).
> La GPU no acelera significativamente estos modelos, pero la instancia GPU
> de Colab Pro+ viene con más RAM y mejor CPU, que es lo que realmente importa.

### Reanudación automática
Si Colab se desconecta, simplemente **reabrí el notebook y ejecutá todas las celdas**.
La DB de Optuna se respalda periódicamente en Google Drive y se restaura automáticamente.
Optuna retoma desde el último trial completado.

In [ ]:
#@title **Configuración** { display-mode: "form" }

PACKAGE = "pes_dqn" #@param ["pes_dqn", "pes_ac"]
N_TRIALS = 110 #@param {type:"integer"}
RESUME_DATE = "2026-03-07" #@param {type:"string"}
BRANCH = "dqn_and_ac" #@param {type:"string"}
DRIVE_SYNC_MINUTES = 5 #@param {type:"slider", min:1, max:30, step:1}
GITHUB_TOKEN = "" #@param {type:"string"}

# --- Derivados ---
GITHUB_REPO = 'https://github.com/Maximiliano0/mPES_2026.git'
_MODULE_MAP = {'pes_dqn': 'optimize_dqn', 'pes_ac': 'optimize_ac'}
OPT_MODULE = _MODULE_MAP[PACKAGE]
REPO_DIR = '/content/mPES'
DRIVE_BACKUP = f'/content/drive/MyDrive/mPES_backups/{PACKAGE}'
DB_SUBDIR = f'{PACKAGE}/inputs/{RESUME_DATE}_BAYESIAN_OPT'
DB_NAME = f'optuna_study_{RESUME_DATE}.db'

import os
os.makedirs(DRIVE_BACKUP, exist_ok=True)

print(f'Paquete:       {PACKAGE}')
print(f'Módulo:        {PACKAGE}.ext.{OPT_MODULE}')
print(f'Trials:        {N_TRIALS}')
print(f'Fecha resume:  {RESUME_DATE}')
print(f'Rama:          {BRANCH}')
print(f'Backup Drive:  {DRIVE_BACKUP}')

## 1. Setup del entorno

In [ ]:
import os, shutil

# --- Montar Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Clonar repositorio ---
if not os.path.exists(REPO_DIR):
    repo_url = GITHUB_REPO
    if GITHUB_TOKEN:
        repo_url = GITHUB_REPO.replace('https://', f'https://{GITHUB_TOKEN}@')
    !git clone --branch {BRANCH} {repo_url} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# --- Restaurar DB desde Drive si es más reciente ---
drive_db = os.path.join(DRIVE_BACKUP, DB_NAME)
repo_db = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)

if os.path.exists(drive_db) and os.path.exists(repo_db):
    if os.path.getmtime(drive_db) > os.path.getmtime(repo_db):
        shutil.copy2(drive_db, repo_db)
        print('DB restaurada desde Drive (más reciente)')
    else:
        print('DB del repo está actualizada')
elif os.path.exists(drive_db):
    os.makedirs(os.path.dirname(repo_db), exist_ok=True)
    shutil.copy2(drive_db, repo_db)
    print('DB restaurada desde Drive')
elif os.path.exists(repo_db):
    shutil.copy2(repo_db, drive_db)
    print('DB del repo copiada a Drive como backup inicial')
else:
    print('No se encontró DB — se creará una nueva')

# --- Instalar dependencias faltantes ---
!pip install -q optuna==4.7.0 gym==0.26.1 pygame==2.5.2 rich==14.3.2 mne==1.6.1 colorlog colorama

# --- Verificar ---
!cd {REPO_DIR} && python3 -c "import optuna, gym, tensorflow as tf; gpus = tf.config.list_physical_devices('GPU'); print(f'TF={tf.__version__}, Optuna={optuna.__version__}, GPU={gpus}')"

In [ ]:
!nvidia-smi

import os, sqlite3

# --- RAM del sistema ---
with open('/proc/meminfo') as f:
    for line in f:
        if 'MemTotal' in line:
            kb = int(line.split()[1])
            print(f'\nRAM total: {kb / 1024**2:.1f} GB')
            break

# --- Verificar archivos necesarios ---
repo_db = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
csv1 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'sequence_lengths.csv')
csv2 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'initial_severity.csv')

print('\nArchivos requeridos:')
for fpath in [repo_db, csv1, csv2]:
    ok = os.path.exists(fpath)
    print(f'  {"ok" if ok else "FALTA"}  {os.path.relpath(fpath, REPO_DIR)}')

# --- Estado actual de la DB ---
if os.path.exists(repo_db):
    print('\nEstado de trials en la DB:')
    conn = sqlite3.connect(repo_db)
    for row in conn.execute("SELECT state, COUNT(*) FROM trials GROUP BY state"):
        print(f'  {row[0]}: {row[1]}')
    best = conn.execute(
        "SELECT MAX(tv.value) FROM trial_values tv "
        "JOIN trials t ON tv.trial_id=t.trial_id WHERE t.state='COMPLETE'"
    ).fetchone()
    if best and best[0]:
        print(f'  Mejor valor: {best[0]:.6f}')
    conn.close()

## 2. Ejecutar optimización

Esta celda lanza la optimización como subproceso con **GPU habilitada**.
Un hilo en segundo plano respalda la DB en Google Drive cada N minutos.

Si Colab se desconecta, volvé a ejecutar todo el notebook desde la celda 1 —
la DB se restaura desde Drive y Optuna retoma automáticamente.

In [ ]:
import subprocess, os, threading, time, shutil

db_src = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
db_dst = os.path.join(DRIVE_BACKUP, DB_NAME)

# --- Hilo de sincronización con Drive ---
done_event = threading.Event()

def sync_to_drive():
    """Copia la DB a Drive periódicamente como respaldo."""
    while not done_event.is_set():
        done_event.wait(DRIVE_SYNC_MINUTES * 60)
        if done_event.is_set():
            break
        try:
            shutil.copy2(db_src, db_dst)
            ts = time.strftime('%H:%M:%S')
            print(f'\n  [Drive Sync {ts}] DB respaldada\n', flush=True)
        except Exception as e:
            print(f'\n  [Drive Sync] Error: {e}\n', flush=True)

sync_thread = threading.Thread(target=sync_to_drive, daemon=True)
sync_thread.start()

# --- Variables de entorno ---
# CUDA_VISIBLE_DEVICES='0' anula el setdefault('-1') del código fuente
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
env['VIRTUAL_ENV'] = '/content'
env['PYTHONUNBUFFERED'] = '1'
env['TF_ENABLE_ONEDNN_OPTS'] = '0'
env['TF_CPP_MIN_LOG_LEVEL'] = '2'

cmd = [
    'python3', '-m', f'{PACKAGE}.ext.{OPT_MODULE}',
    str(N_TRIALS), '--resume', RESUME_DATE
]

sep = '=' * 60
print(sep)
print(f'  {" ".join(cmd)}')
print(f'  cwd: {REPO_DIR}')
print(f'  GPU: CUDA_VISIBLE_DEVICES=0')
print(f'  Drive sync: cada {DRIVE_SYNC_MINUTES} min')
print(sep + '\n')

t_start = time.time()
proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

for line in proc.stdout:
    print(line, end='', flush=True)

exit_code = proc.wait()
done_event.set()
elapsed = time.time() - t_start

# --- Backup final ---
shutil.copy2(db_src, db_dst)

print(f'\n{sep}')
print(f'  Proceso finalizado (código: {exit_code})')
print(f'  Tiempo total: {elapsed/3600:.1f}h ({elapsed/60:.0f} min)')
print(f'  DB respaldada en: {db_dst}')
print(sep)

## 3. Resultados

In [ ]:
import sqlite3, os, shutil

db = os.path.join(REPO_DIR, DB_SUBDIR, DB_NAME)
conn = sqlite3.connect(db)

print('=' * 50)
print('  RESULTADOS DE LA OPTIMIZACIÓN')
print('=' * 50)

print('\nTrials por estado:')
for row in conn.execute("SELECT state, COUNT(*) FROM trials GROUP BY state"):
    print(f'  {row[0]}: {row[1]}')

print('\nTop 10 trials:')
rows = conn.execute(
    "SELECT t.trial_id, tv.value "
    "FROM trial_values tv JOIN trials t ON tv.trial_id=t.trial_id "
    "WHERE t.state='COMPLETE' ORDER BY tv.value DESC LIMIT 10"
).fetchall()
for i, row in enumerate(rows, 1):
    marker = ' (BEST)' if i == 1 else ''
    print(f'  #{row[0]:3d}  {row[1]:.6f}{marker}')

conn.close()

# --- Copiar todos los outputs a Drive ---
opt_dir_src = os.path.join(REPO_DIR, DB_SUBDIR)
opt_dir_dst = os.path.join(DRIVE_BACKUP, f'{RESUME_DATE}_BAYESIAN_OPT')
if os.path.exists(opt_dir_src):
    shutil.copytree(opt_dir_src, opt_dir_dst, dirs_exist_ok=True)
    print(f'\nOutputs copiados a: {opt_dir_dst}')
    for fname in sorted(os.listdir(opt_dir_dst)):
        size = os.path.getsize(os.path.join(opt_dir_dst, fname))
        print(f'  {fname} ({size/1024:.0f} KB)')

In [ ]:
#@title **Opcional: Push a GitHub** { display-mode: "form" }

if GITHUB_TOKEN:
    !cd {REPO_DIR} && git add -A && git commit -m "colab: {PACKAGE} optimización ({RESUME_DATE})" && git push
    print('Cambios pusheados a GitHub')
else:
    print('Configurá GITHUB_TOKEN en la celda de configuración para hacer push.')
    print(f'Alternativa: descargá la DB desde Google Drive:')
    print(f'  {DRIVE_BACKUP}')